In [1]:
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')


import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if

#from visualization_system.visualization_backend.analyst.analyst_backend import * 
#from visualization_system.visualization_backend.analyst.analyst_backend import *

from visualization_system.visualization_backend.analyst.semantics.semantic_models import SemanticCatalog, semantic_catalog
from visualization_system.visualization_backend.analyst.semantics.semantic_models import idioms as all_idiom_rules



imported


In [2]:
type(semantic_catalog)



dict

# Initialize 

In [19]:
#from visualization_system.visualization_backend.analyst.analyst_backend import VisualizationAgenticSystem


In [3]:

# these are just mocks 


from visualization_system.visualization_backend.analyst.analyst_backend import VisualizationAgenticSystem
 
def get_config():
    return None 

class DataDrivenStorage:
        
    def __init__( self, config_vars ):
        pass 

    def get_project_dataset(self, project_name=None, filters=None):
        #path =  "../datasets/Demo1/"
        path =  Path("../datasets/IX5I_4P/") 

        inj, prod, locs = self.fetch_data(path) 
        return inj, prod, locs

    def fetch_data(self,path:Path):
        inj  = pd.read_csv(path / "injectors.csv")
        pinj = pd.read_csv(path / "producers.csv")
        locs = pd.read_csv(path / "locations.csv")
        inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
        inj['DAY']   = inj['DATE'].dt.day
        inj['MONTH'] = inj['DATE'].dt.month
        inj['YEAR']  = inj['DATE'].dt.year
        pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
        pinj['DAY']   = pinj['DATE'].dt.day
        pinj['MONTH'] = pinj['DATE'].dt.month
        pinj['YEAR']  = pinj['DATE'].dt.year


        return inj, pinj, locs

def init_visualization_system( llm ):
   

    vis_system = VisualizationAgenticSystem( llm )


    idiom = 'duckdb'
    idiom_rules = all_idiom_rules[idiom]
    semantic_catalog_model = SemanticCatalog.model_validate( semantic_catalog )

    print( type(semantic_catalog_model))

    analyst = vis_system.data_analyst_component
    analyst.init_semantic_models( semantic_catalog_model,idiom_rules)
    
    print(analyst.smart_data.catalog_snapshot())



    return vis_system


llm = azure_llm_if()
vis_system = init_visualization_system(llm)


# data changes
# this mocks data comming from the UI
# so we just update tge analyst 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
vis_system.data_analyst_component.set_data( {'injectors':inj, 
                                             'producers':prod, 
                                             'locations': locs } )





imported
zero temp, seed 42, top_p = 1
getting some tools here 
<class 'visualization_system.visualization_backend.analyst.semantics.semantic_models.SemanticCatalog'>
base_tables=[TableCard(name='injectors', description='Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', kind='base', creation_date='2026-09-11 23:38:36.169328', row_count=None, columns=[ColumnCard(name='DATE', data_type='timestamp', description='Injection date.', derived_column=False), ColumnCard(name='NAME', data_type='string', description='Injector well identifier.', derived_column=False), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume.', derived_column=False), ColumnCard(name='SUBZONE', data_type='string', description='Vertical subzone.', derived_column=False), ColumnCard(name='SECTOR', data_type='integer', description='Geographic sector.', derived_column=False), ColumnCard(name

In [4]:

query2 = """
List the 5 wells with the highest water cut in current date
"""



#this is what the presenter consumes 
execution_state = vis_system.run( query2 )


Plan created
Routing next task: task_index_to_execute=0, total tasks=1
Running the sql analyst
base_tables=[TableCard(name='injectors', description='Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', kind='base', creation_date='2026-09-11 23:38:36.207388', row_count=490, columns=[ColumnCard(name='DATE', data_type='timestamp', description='Injection date.', derived_column=False), ColumnCard(name='NAME', data_type='string', description='Injector well identifier.', derived_column=False), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume.', derived_column=False), ColumnCard(name='SUBZONE', data_type='string', description='Vertical subzone.', derived_column=False), ColumnCard(name='SECTOR', data_type='integer', description='Geographic sector.', derived_column=False), ColumnCard(name='YEAR', data_type='integer', description='Year from DATE.', derived_colu

In [5]:
vis_system.data_analyst_component.smart_data.catalog_snapshot()

CatalogTablesSnapshot(base_tables=[TableCard(name='injectors', description='Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', kind='base', creation_date='2026-09-11 23:38:36.207388', row_count=490, columns=[ColumnCard(name='DATE', data_type='timestamp', description='Injection date.', derived_column=False), ColumnCard(name='NAME', data_type='string', description='Injector well identifier.', derived_column=False), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume.', derived_column=False), ColumnCard(name='SUBZONE', data_type='string', description='Vertical subzone.', derived_column=False), ColumnCard(name='SECTOR', data_type='integer', description='Geographic sector.', derived_column=False), ColumnCard(name='YEAR', data_type='integer', description='Year from DATE.', derived_column=False), ColumnCard(name='MONTH', data_type='integer', description='Mon

In [6]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
print(ui_items)

for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 




Processing task result from agent: agent='data_analysis' instruction='Determine the 5 wells with the highest water cut on the most recent date available in the dataset.' cheap_output='data_analysis agent created and stored the table `top_5_wells_highest_water_cut`: This table contains the top 5 wells with the highest water cut on the most recent date available in the dataset. The water cut is calculated as the ratio of produced water volume to total produced liquid volume.\n\nTable `top_5_wells_highest_water_cut` contents:\nNAME       DATE  water_cut\n  P1 2023-12-31   0.867770\n  P3 2023-12-31   0.822109\n  P2 2023-12-31   0.820708\n  P4 2023-12-31   0.781266' raw_results=[AgentTableResponse(agent='analyst', clarification=None, user_query='Determine the 5 wells with the highest water cut on the most recent date available in the dataset.', tables=[TableItemAgentResponse(table_name='top_5_wells_highest_water_cut', description='This table contains the top 5 wells with the highest water c

In [7]:
query2 = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

planner = vis_system.planner_component
plan = planner.run( query2 )

In [8]:
pprint.pprint( plan.model_dump())


{'clarification_request': None,
 'direct_answer': None,
 'tasks': [{'agent': 'direct_answer',
            'instruction': 'Explain the concept of Voidage Replacement Ratio '
                           '(VRR) in reservoir engineering.'},
           {'agent': 'data_analysis',
            'instruction': 'Identify the top 5 producer wells based on their '
                           'cumulative oil production in the year 2018.'},
           {'agent': 'data_analysis',
            'instruction': 'Calculate and display the cumulative liquid '
                           'production for all wells starting from the year '
                           '2015.'}],
 'user_intent': 'Explain VRR, identify top 5 producers by cumulative oil '
                'production in 2018, and show cumulative liquid production '
                'since 2015 for all wells.'}


'Provide a brief explanation of the concept of VRR (Voidage Replacement Ratio).'

In [9]:
direct_answer = vis_system.direct_answer_component

direct_answer.run(  plan.tasks[0].instruction  ).data_results

[TextResult(text='The Voidage Replacement Ratio (VRR) is a key concept in reservoir engineering that measures the balance between the volume of fluids injected into a reservoir and the volume of fluids produced. It is defined as:\n\n**VRR = Volume of Fluids Injected / Volume of Fluids Produced**\n\n- **Purpose**: VRR is used to maintain reservoir pressure and optimize hydrocarbon recovery. A VRR of 1.0 indicates that the volume of injected fluids (e.g., water, gas) equals the volume of produced fluids (oil, gas, and water), helping to sustain pressure and improve sweep efficiency.\n\n- **Applications**: \n  - In waterflooding or gas injection projects, maintaining a VRR close to 1.0 is often desirable to prevent pressure depletion or over-pressurization.\n  - In reservoirs with strong aquifer support, a VRR less than 1.0 may be acceptable, as the aquifer contributes to pressure maintenance.\n\n- **Considerations**: The calculation must account for fluid compressibility, reservoir condi

In [11]:

analyst = vis_system.data_analyst_component
task_results = [ analyst.run(task.instruction) for task in plan.tasks[1:] ]



base_tables=[TableCard(name='injectors', description='Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', kind='base', creation_date='2026-09-11 23:38:36.207388', row_count=490, columns=[ColumnCard(name='DATE', data_type='timestamp', description='Injection date.', derived_column=False), ColumnCard(name='NAME', data_type='string', description='Injector well identifier.', derived_column=False), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume.', derived_column=False), ColumnCard(name='SUBZONE', data_type='string', description='Vertical subzone.', derived_column=False), ColumnCard(name='SECTOR', data_type='integer', description='Geographic sector.', derived_column=False), ColumnCard(name='YEAR', data_type='integer', description='Year from DATE.', derived_column=False), ColumnCard(name='MONTH', data_type='integer', description='Month from DATE.', derive

In [38]:
task_results

[TaskResult(agent='data_analysis', instruction='Identify the top 5 producer wells based on their cumulative oil production in the year 2018.', cheap_output='data_analysis agent created and stored the table `top_5_producer_wells_2018`: This table contains the top 5 producer wells based on their cumulative oil production in the year 2018. Columns include well name and cumulative oil production volume.\n\nTable `top_5_producer_wells_2018` contents:\nNAME  cumulative_oil_volume\n  P4           12875.199035\n  P3           12768.234316\n  P1           12538.622009\n  P2           11907.329712', raw_results=[AgentTableResponse(agent='analyst', clarification=None, user_query='Identify the top 5 producer wells based on their cumulative oil production in the year 2018.', tables=[TableItemAgentResponse(table_name='top_5_producer_wells_2018', description='This table contains the top 5 producer wells based on their cumulative oil production in the year 2018. Columns include well name and cumulativ

In [42]:
from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = [ presenter.process_single_task_result(result)[0] for result in task_results ] 





Processing dataframe result with shape: (4, 2)
  NAME  cumulative_oil_volume
0   P4           12875.199035
1   P3           12768.234316
2   P1           12538.622009
3   P2           11907.329712
****chart plan****
Identify the top 5 producer wells based on their cumulative oil production in the year 2018.
{'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'cumulative_oil_volume', 'aggregate': None, 'group_by': ['NAME'], 'color_by': None, 'orientation': 'v', 'barmode': 'group', 'title': 'Top 5 Producer Wells by Cumulative Oil Production in 2018'}}}
Processing dataframe result with shape: (4, 2)
  NAME  cumulative_liquid_volume
0   P1             136730.784668
1   P4             119989.718749
2   P3             127573.088439
3   P2             129323.377559
****chart plan****
Calculate and display the cumulative liquid production for all wells starting from the year 2015.
{'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'cum

In [43]:
ui_items

[UIItem(id='chart_86c97235', type='chart', title='Top 5 producer wells 2018', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'bar', 'name': 'Cumulative oil volume', 'orientation': 'v', 'x': ['P4', 'P3', 'P1', 'P2'], 'y': [12875.1990353, 12768.2343158, 12538.622009199999, 11907.329711700002]}], 'layout': {'title': {'text': 'Top 5 producer wells by cumulative oil production in 2018'}, 'xaxis': {'title': {'text': 'Name'}}, 'yaxis': {'title': {'text': 'Cumulative oil volume'}}, 'barmode': 'group'}, 'config': {'responsive': True, 'displaylogo': False}}}, meta={'description': 'This table contains the top 5 producer wells based on their cumulative oil production in the year 2018. Columns include well name and cumulative oil production volume.'}),
 UIItem(id='chart_88682a88', type='chart', title='Cumulative liquid production from 2015', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'bar', 'name': 'Cumulative liquid volume', 'orientation': 'v', 'x': ['P1', 'P4', 'P3', 'P2'], 'y

In [44]:


for item in ui_items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 


In [45]:
query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production for each year since year 2015 for all the wells 
"""

state = vis_system.run( query )


Plan created
Routing next task: task_index_to_execute=0, total tasks=3
Routing next task: task_index_to_execute=1, total tasks=3
Running the sql analyst
base_tables=[TableCard(name='injectors', description='Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', kind='base', creation_date='2026-08-21 14:55:53.423777', row_count=490, columns=[ColumnCard(name='DATE', data_type='timestamp', description='Injection date.', derived_column=False), ColumnCard(name='NAME', data_type='string', description='Injector well identifier.', derived_column=False), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume.', derived_column=False), ColumnCard(name='SUBZONE', data_type='string', description='Vertical subzone.', derived_column=False), ColumnCard(name='SECTOR', data_type='integer', description='Geographic sector.', derived_column=False), ColumnCard(name='YEAR', data_t

In [46]:
presenter = PresenterComponent4( llm )

ui_items = presenter.run(state)

Processing task result from agent: agent='direct_answer' instruction='Provide a brief explanation of the concept of Voidage Replacement Ratio (VRR).' cheap_output='The Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the pressure in the reservoir is being maintained effectively during production. \n\nMathematically, VRR is expressed as:\n\n**VRR = Volume of Injected Fluids / Volume of Produced Fluids**\n\nA VRR of 1 indicates that the injected fluid volume equals the produced fluid volume, helping maintain reservoir pressure. A VRR less than 1 suggests pressure depletion, while a VRR greater than 1 may indicate over-injection, potentially leading to operational issues like water breakthrough.' raw_results=['The Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the r

In [50]:
ui_items.items

[UIItem(id='text_60f63463', type='text', title=None, data={'text': 'The Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the pressure in the reservoir is being maintained effectively during production. \n\nMathematically, VRR is expressed as:\n\n**VRR = Volume of Injected Fluids / Volume of Produced Fluids**\n\nA VRR of 1 indicates that the injected fluid volume equals the produced fluid volume, helping maintain reservoir pressure. A VRR less than 1 suggests pressure depletion, while a VRR greater than 1 may indicate over-injection, potentially leading to operational issues like water breakthrough.'}, meta={}),
 UIItem(id='chart_14b93f2d', type='chart', title='Top 5 producer wells 2018', data={'engine': 'plotly', 'plotly': {'data': [{'name': 'Cumulative oil volume', 'orientation': 'v', 'x': ['P4', 

In [48]:
for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 

The Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the pressure in the reservoir is being maintained effectively during production. 

Mathematically, VRR is expressed as:

**VRR = Volume of Injected Fluids / Volume of Produced Fluids**

A VRR of 1 indicates that the injected fluid volume equals the produced fluid volume, helping maintain reservoir pressure. A VRR less than 1 suggests pressure depletion, while a VRR greater than 1 may indicate over-injection, potentially leading to operational issues like water breakthrough.


In [51]:
state 

{'user_query': 'Explain VRR briefly and then \nlist the 5 top producers based on the cummulated oil production in 2018,\nthen show the cummulated liquid production for each year since year 2015 for all the wells \n',
 'plan': VisualizationSystemPlan(user_intent='Understand VRR and analyze production data for top producers and yearly liquid production.', tasks=[VisualizationSystemTask(instruction='Provide a brief explanation of the concept of Voidage Replacement Ratio (VRR).', agent='direct_answer'), VisualizationSystemTask(instruction='Identify the top 5 producer wells based on their cumulative oil production in the year 2018.', agent='data_analysis'), VisualizationSystemTask(instruction='Calculate the cumulative liquid production for each year since 2015 for all wells.', agent='data_analysis')], clarification_request=None, direct_answer=None),
 'task_index_to_execute': 3,
 'facts_context': 'The Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the rati

In [39]:
from visualization_system.visualization_backend.analyst.analyst_backend import DataFrameResult, ExecutorState, PresenterConfig, PresenterResponse, SubInstructions, TableResponseProcessor, TaskResult, TextResult, UIItem
  


In [40]:
from dataclasses import dataclass
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')
import warnings

from typing import Any, Dict, Generic, List, Iterable, Literal, TypeVar, Union, Optional,TypedDict
from typing_extensions import Self
   
from uuid import uuid4
 
from pydantic import BaseModel, Field 
from visualization_system.visualization_backend.get_llm_model import azure_llm_if
from pathlib import Path
from langchain.agents.structured_output import ToolStrategy
from langgraph.graph import StateGraph, END

from langchain.agents import create_agent

from visualization_system.common.base_plan import PlannerConfig
#from visualization_system.visualization_backend.analyst.analyst_system import PlannerConfig, DirectAnswerConfig
from visualization_system.visualization_backend.analyst.analyst_models import TableItemAgentResponse, VisualizationSystemPlanner, VisualizationSystemTask, VisualizationSystemPlan 
from visualization_system.visualization_backend.analyst.smart_data import SmartData
from visualization_system.visualization_backend.analyst.smart_data_tools import SmartDataTools
from visualization_system.visualization_backend.analyst.analyst_system import SQLAnalystConfig

from visualization_system.visualization_backend.analyst.prompts import visualization_planner_prompt3 
from visualization_system.visualization_backend.analyst.prompts import anayst_prompt_template 
from visualization_system.visualization_backend.analyst.prompts import chart_agent_prompt
from visualization_system.visualization_backend.analyst.prompts import small_table_prompt
from visualization_system.visualization_backend.analyst.prompts import split_subinstructions_prompt

 


import pandas as pd, re, json  
from langchain_core.messages import SystemMessage, HumanMessage
from visualization_system.visualization_backend.global_models import UIState



In [ ]:
from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4

CHART_AGENT_PROMPTV4 = """
You are a chart planning agent.

You receive:
- user query
- table summaries
- column names, roles, cardinality, and descriptions

Return a JSON plan with:
- zero or more ordered preprocess operations 
- exactly ONE plot step. 
- ONE plot step (see <plot_tools> below) must be in the plan regardless of whether there are or not preprocess operations

 
PREPROCESSING

The preprocess field is an ordered list of operations applied before plotting.

Supported operations:

1. create_combined_category
   Creates a category column from two existing columns.

Args:
{
"operation": "create_combined_category",
"args": {
"col1": "<column>",
"col2": "<column>",
"new_col": "<new_column>",
"sep": " / "
}
}

2. create_date_bucket
   Creates a date grouping column.

Args:
{
"operation": "create_date_bucket",
"args": {
"date_col": "<date_column>",
"bucket": "D|W|M|Q|Y",
"new_col": "<new_column>"
}
}

3. filter_rows
   Keeps rows matching one or more conditions.

Args:
{
"operation": "filter_rows",
"args": {
"filters": [
{
"column": "<column>",
"operator": "==|!=|>|>=|<|<=|in|not_in",
"value": "<value_or_list>"
}
]
}
}

4. aggregate
   Groups and aggregates the data before plotting.

Args:
{
"operation": "aggregate",
"args": {
"group_by": ["<column>", "..."],
"metrics": {
"<numeric_column>": "sum|mean|median|min|max|count|nunique"
}
}
}

5. sort_rows
   Sorts the rows.

Args:
{
"operation": "sort_rows",
"args": {
"sort_by": "<column_or_list>",
"ascending": true|false
}
}

6. limit_rows
   Keeps only the first N rows.

Args:
{
"operation": "limit_rows",
"args": {
"n": <integer>
}
}

7. select_columns
   Keeps only selected columns.

Args:
{
"operation": "select_columns",
"args": {
"columns": ["<column>", "..."]
}
}

8. select_top_entities
   Selects the top or bottom entities using a metric.

Use keep_all_rows = true when the ranking period is only used to identify entities, but the final chart needs all rows for those entities.

Args:
{
"operation": "select_top_entities",
"args": {
"entity_col": "<entity_column>",
"metric_col": "<numeric_column>",
"n": <integer>,
"aggregate": "sum|mean|median|min|max|count|nunique",
"ascending": true|false,
"filters": [],
"keep_all_rows": true|false
}
}

Rules:

* Use preprocess only when the input table is not already ready for plotting.
* Operations are executed in the listed order.
* Do not invent columns.
* Prefer the smallest number of operations needed.
* Aggregation, filtering, ranking, date bucketing and limiting should be done in preprocess rather than in the plotting tool.



PLOT TOOLS



Args:
{
  "columns": ["<column>", "..."],
  "sort_by": null | "<column>",
  "sort_order": "asc|desc",
  "limit": null | <integer>,
  "title": "<title>"
}


plot_bar_chart:
Use for comparing one or more quantitative values across categorical or bucketed temporal groups.

Best for:
- "Y by A"
- "Y per A"
- "Y by A and B"
- totals, averages, counts, rankings, grouped comparisons

Mapping rules:
- For "Y by A": use x = A, y = Y, group_by = [A].
- For "Y by A and B": use x = A, color_by = B, y = Y, group_by = [A, B].
- For "Y by A, B, and C": use x = A, color_by = B or C, and group_by = [A, B, C].
- If two columns together define the x-axis label, create the combined column first with preprocess_for_chart and use it as x.
- group_by must include every column needed to preserve the requested breakdown.
- Use aggregate = "sum" by default for additive quantities unless the query specifies another aggregation.

Do not use a bar chart for multi-period time-series trends when a line chart can show the evolution more clearly.

A temporal column does not automatically make a bar chart appropriate.
Use bars for discrete period totals only when the user explicitly asks to compare independent periods or requests a bar chart.

Args:
{
  "x": "<category_or_bucket_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<secondary_category_col>",
  "orientation": "v|h",
  "barmode": "group|stack|relative",
  "title": "<title>"
}

plot_line_chart:
Use for trends, time series, ordered progression, or cumulative values over time.

Use a line chart when:
- x is a date, year, month, quarter, or another ordered temporal column;
- the user asks for yearly, monthly, quarterly, or daily evolution;
- the chart shows how a metric changes across multiple time periods;
- multiple entities should be represented as separate time-series traces.

For "Y by time for each A":
- x = time column
- y = Y
- series_by = A
- each unique series_by value becomes one trace

Prefer a line chart over a bar chart whenever the main purpose is to show change or evolution over time.

Trace rules:
- Use series_by when one column defines separate traces.
- series_by values become the trace names.
- Use series_by = NAME when each well should be a separate trace.
- If y is a list and series_by is provided, traces are named "<series_by value> - <y column>".
- If series_by is null, traces are named from y column names.
- color_by is deprecated. Use series_by instead.

Args:
{
  "x": "<time_or_ordered_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>", "..."],
  "series_by": null | "<category_col>",
  "date_bucket": null | "D|W|M|Q|Y",
  "cumulative": true|false,
  "title": "<title>"
}

plot_pie_chart:
Use only for part-to-whole/share/composition questions.
Args:
{
  "labels": "<category_col>",
  "values": "<numeric_col>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<label_col>"],
  "hole": 0.0,
  "title": "<title>"
}

plot_scatter_chart:
Use for numeric-vs-numeric relationships, correlations, crossplots, clusters, or row-level comparisons.
Args:
{
  "x": "<numeric_col>",
  "y": "<numeric_col_or_list>",
  "series_by": null | "<category_col>",
  "size_by": null | "<numeric_col>",
  "text_by": null | "<label_col>",
  "title": "<title>"
}

IMPORTANT
YOU MUST address only the parts of the user question for which the table is related
YOU MUST Ignore the parts of the question that the information in the table cannot address
             

RULES
- Return only valid JSON.
- Do not invent tools.
- Do not invent arguments.
- Use only columns that exist or are created by preprocess_for_chart.
- Prefer no preprocess when existing columns are sufficient.
- Use sum by default for additive quantities unless otherwise specified.
- If uncertain, return {"reason": "...", "preprocess": null, "plot": null}.
- When multiple temporal dimensions together define the displayed x-axis grouping
(e.g. year + quarter, year + month),
create a combined temporal category for x.

OUTPUT SHAPE
{

  "preprocess": [],
  "plot": {
    "tool": "<plot_tool>",
    "args": {}
  }
}
"""

chart_agent_prompt2 = CHART_AGENT_PROMPTV4

c = PresenterConfig( )
c.prompt = chart_agent_prompt 

In [ ]:
query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

planner = vis_system.planner_component
plan = planner.run( query )


In [41]:


query2 = """Explain VRR briefly and then:

1. plot the yearly liquid production of the 5 top producers based on the cummulated oil production in 2018, 
2. show the cummulated liquid production since year 2015 for the first two of those wells.  

"""


#this is what the presenter consumes 
execution_state = vis_system.run( query2 )



base_tables=[TableCard(name='injectors', description='Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', kind='base', creation_date='2026-08-10 12:52:30.901284', row_count=490, columns=[ColumnCard(name='DATE', data_type='timestamp', description='Injection date.', derived_column=False), ColumnCard(name='NAME', data_type='string', description='Injector well identifier.', derived_column=False), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume.', derived_column=False), ColumnCard(name='SUBZONE', data_type='string', description='Vertical subzone.', derived_column=False), ColumnCard(name='SECTOR', data_type='integer', description='Geographic sector.', derived_column=False), ColumnCard(name='YEAR', data_type='integer', description='Year from DATE.', derived_column=False), ColumnCard(name='MONTH', data_type='integer', description='Month from DATE.', derive

In [54]:
import pickle
with open("execution_state.pkl", "wb") as file:
    pickle.dump(execution_state, file)


In [3]:
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data

In [4]:
execution_state

{'user_query': '\nList the 5 wells with the highest water cut in current date\n',
 'plan': VisualizationSystemPlan(user_intent='Identify the top 5 wells with the highest water cut on the most recent date in the dataset.', tasks=[VisualizationSystemTask(instruction='Determine the 5 wells with the highest water cut on the most recent date available in the dataset. Provide the well names and their respective water cut values.', agent='data_analysis')], clarification_request=None, direct_answer=None),
 'task_index_to_execute': 1,
 'facts_context': 'data_analysis agent created and stored the table `top_5_wells_highest_water_cut`: This table contains the top 5 wells with the highest water cut values on the most recent date available in the dataset. It includes the well identifier (NAME) and the calculated water cut (water_cut).\n\nTable `top_5_wells_highest_water_cut` contents:\nNAME  water_cut\n  P1   0.867770\n  P3   0.822109\n  P2   0.820708\n  P4   0.781266',
 'cheap_tool_outputs': ['dat

In [ ]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
print(ui_items)

for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 




In [ ]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
ui_items

Processing task result from agent: agent='data_analysis' instruction='Determine the 5 wells with the highest water cut on the most recent date available in the dataset. Provide the well names and their respective water cut values.' cheap_output='data_analysis agent created and stored the table `top_5_wells_highest_water_cut`: This table contains the top 5 wells with the highest water cut values on the most recent date available in the dataset. It includes the well identifier (NAME) and the calculated water cut (water_cut).\n\nTable `top_5_wells_highest_water_cut` contents:\nNAME  water_cut\n  P1   0.867770\n  P3   0.822109\n  P2   0.820708\n  P4   0.781266' raw_results=[AgentTableResponse(agent='analyst', clarification=None, user_query='Determine the 5 wells with the highest water cut on the most recent date available in the dataset. Provide the well names and their respective water cut values.', tables=[TableItemAgentResponse(table_name='top_5_wells_highest_water_cut', description='Th

PresenterResponse(agent='presenter', layout='vertical', items=[UIItem(id='chart_ee951b73', type='chart', title='Top 5 wells highest water cut', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'table', 'header': {'values': ['Name', 'Water cut'], 'align': 'left'}, 'cells': {'values': [['P1', 'P3', 'P2', 'P4'], ['0.8677698548904886', '0.8221089403073', '0.8207081193550646', '0.7812662315549471']], 'align': 'left'}}], 'layout': {'title': {'text': 'Top 5 wells with highest water cut on the most recent date'}}, 'config': {'responsive': True, 'displaylogo': False}}}, meta={'description': 'This table contains the top 5 wells with the highest water cut values on the most recent date available in the dataset. It includes the well identifier (NAME) and the calculated water cut (water_cut).'})])

In [7]:
ui_items.items

[UIItem(id='chart_4fc653e5', type='chart', title='Top 5 wells highest water cut', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'table', 'header': {'values': ['Name', 'Water cut'], 'align': 'left'}, 'cells': {'values': [['P1', 'P3', 'P2', 'P4'], ['0.8677698548904886', '0.8221089403073', '0.8207081193550646', '0.7812662315549471']], 'align': 'left'}}], 'layout': {'title': {'text': 'Top 5 wells with highest water cut on the most recent date'}}, 'config': {'responsive': True, 'displaylogo': False}}}, meta={'description': 'This table contains the top 5 wells with the highest water cut values on the most recent date available in the dataset. It includes the well identifier (NAME) and the calculated water cut (water_cut).'})]

In [56]:
for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 
        

# END 

In [ ]:
#item = ui_items.items[1]
#item = item.data['plotly']
#pio.show(item)

item = ui_items.items[3]
item = item.data['plotly']
pio.show(item)




# Improved the presenter. 

In [1]:
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if

from visualization_system.visualization_backend.all_classes import * 



imported


In [2]:
llm = azure_llm_if()
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data


presenter = PresenterComponent4(llm)

presenter_response = presenter.run(execution_state)

ui_items = presenter_response.items

ui_items 

zero temp, seed 42, top_p = 1
****chart plan****
Identify the top 5 producer wells based on their cumulative oil production in 2018.
{'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'total_oil_volume', 'aggregate': None, 'group_by': ['NAME'], 'color_by': None, 'orientation': 'v', 'barmode': 'group', 'title': 'Top 5 Producer Wells by Cumulative Oil Production in 2018'}}}
****chart plan****
Plot the yearly liquid production for the top 5 producer wells.
{'preprocess': [], 'plot': {'tool': 'plot_line_chart', 'args': {'x': 'YEAR', 'y': 'total_liquid_volume', 'aggregate': 'sum', 'group_by': ['YEAR', 'NAME'], 'series_by': 'NAME', 'title': 'Yearly Liquid Production for the Top 5 Producer Wells'}}}
****chart plan****
Calculate and display the cumulative liquid production since 2015 for all wells.
{'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'cumulative_liquid_volume', 'aggregate': None, 'group_by': ['NAME'], 'color_by': None, 

[UIItem(id='text_57f71895', type='text', title=None, data={'text': 'Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. \n\nA VRR of 1.0 indicates full replacement, helping to sustain pressure and improve sweep efficiency, while a VRR less than 1.0 suggests under-replacement, potentially leading to pressure decline. Conversely, a VRR greater than 1.0 may indicate over-injection, which could lead to operational inefficiencies or formation damage.'}, meta={}),
 UIItem(id='chart_2c8f2b7f', type='chart', title='Top 5 producers 2018', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'bar', 'name': 'Total oil volume', 'orientation': 'v', 'x': ['P4', 'P3', 'P1', 'P2'], 'y': [12875.199035

In [3]:
for item in ui_items:
    if item.type=='text':
        print( item.data['text'])
    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. 

A VRR of 1.0 indicates full replacement, helping to sustain pressure and improve sweep efficiency, while a VRR less than 1.0 suggests under-replacement, potentially leading to pressure decline. Conversely, a VRR greater than 1.0 may indicate over-injection, which could lead to operational inefficiencies or formation damage.


# END 

In [ ]:
from visualization_system.visualization_backend.analyst.prompts import chart_agent_prompt
from visualization_system.visualization_backend.analyst.prompts import small_table_prompt
from visualization_system.visualization_backend.analyst.prompts import split_subinstructions_prompt

class SubInstruction(BaseModel):
    """
    One presentation item to produce from one source.
    """

    sub_instruction: str = Field(
        description=(
            "The specific part of the original instruction that this source "
            "should answer."
        )
    )

    kind: Literal["table", "text"] = Field(
        description="Whether the source is a table or a text result."
    )

    source_id: str = Field(
        description=(
            "The exact SOURCE_ID provided in the available sources. "
            "For tables use the table name. "
            "For text use the text SOURCE_ID."
        )
    )


class SubInstructions(BaseModel):
    """
    Ordered presentation plan for one TaskResult.
    """

    items: list[SubInstruction] = Field(
        description=(
            "The ordered list of presentation items to generate."
        )
    )
    
    
class PresenterChartingTools:

    ALLOWED_AGGS = {"sum", "mean", "median", "min", "max", "count", "nunique"}


    plotly_config = {
                "responsive": True,
                "displaylogo": False,
            }



    def run_preprocess(
        self,
        df: pd.DataFrame,
        preprocess_steps: list[dict] | None,
    ) -> pd.DataFrame:
        work = df.copy()

        for step in preprocess_steps or []:
            
            operation = step.get("operation")

            if not operation:
                raise ValueError("Preprocess step is missing 'operation'")

            args = step.get("args") or {}

            work = self.run_preprocess_operation(
                operation=operation,
                df=work,
                args=args,
            )

        return work


    def run_preprocess_operation(
        self,
        operation: str,
        df: pd.DataFrame,
        args: dict[str, Any],
    ) -> pd.DataFrame:
        preprocess_tools = {
            "filter_rows": self.filter_rows,
            "aggregate": self.aggregate_for_chart,
            "sort_rows": self.sort_rows,
            "limit_rows": self.limit_rows,
            "select_columns": self.select_columns,
            "select_top_entities": self.select_top_entities,
            "create_combined_category": self.create_combined_category,
            "create_date_bucket": self.create_date_bucket,
        }

        if operation not in preprocess_tools:
            raise ValueError(f"Unknown preprocess operation: {operation}")

        return preprocess_tools[operation](
            df=df,
            **args,
        )


    ##########################
    #       pre-process      # 
    ##########################
    def filter_rows(self,df: pd.DataFrame,filters: list[dict]) -> pd.DataFrame:
        
        work = df.copy()
        for item in filters:
            column = item["column"]
            operator = item["operator"]
            value = item["value"]

            self._validate_columns(work, [column])

            if operator == "==":
                work = work[work[column] == value]
            elif operator == "!=":
                work = work[work[column] != value]
            elif operator == ">":
                work = work[work[column] > value]
            elif operator == ">=":
                work = work[work[column] >= value]
            elif operator == "<":
                work = work[work[column] < value]
            elif operator == "<=":
                work = work[work[column] <= value]
 


            elif operator == "in":
                if not isinstance(value, (list, tuple, set)):
                    raise ValueError("'in' filter value must be a list")
                work = work[work[column].isin(value)]

            elif operator == "not_in":
                if not isinstance(value, (list, tuple, set)):
                    raise ValueError("'not_in' filter value must be a list")
                work = work[~work[column].isin(value)]




            else:
                raise ValueError(f"Unsupported filter operator: {operator}")

        return work

    def aggregate_for_chart( self, df: pd.DataFrame, group_by: list[str],
        metrics: dict[str, str],
    ) -> pd.DataFrame:
        
        self._validate_columns(df, group_by)

        for column, aggregate in metrics.items():
            self._validate_columns(df, [column])

            if aggregate not in self.ALLOWED_AGGS:
                raise ValueError(f"Unsupported aggregate: {aggregate}")

        return (df.groupby(group_by, dropna=False, as_index=False).agg(metrics))

    def sort_rows(
        self,
        df: pd.DataFrame,
        sort_by: str | list[str],
        ascending: bool = True,
    ) -> pd.DataFrame:
        sort_columns = self._as_list(sort_by)
        self._validate_columns(df, sort_columns)

        return df.sort_values(
            sort_columns,
            ascending=ascending,
        )

    def limit_rows(
        self,
        df: pd.DataFrame,
        n: int,
    ) -> pd.DataFrame:
        return df.head(n)

    def select_columns(
        self,
        df: pd.DataFrame,
        columns: list[str],
    ) -> pd.DataFrame:
        self._validate_columns(df, columns)
        return df[columns].copy()

    def create_combined_category(
        self,
        df: pd.DataFrame,
        col1: str,
        col2: str,
        new_col: str | None = None,
        sep: str = " / ",
    ) -> pd.DataFrame:
        out = df.copy()

        self._validate_columns(out, [col1, col2])

        new_col = new_col or f"{col1}_{col2}"

        out[new_col] = (
            out[col1].fillna("").astype(str)
            + sep
            + out[col2].fillna("").astype(str)
        )

        return out

    def create_date_bucket(
        self,
        df: pd.DataFrame,
        date_col: str,
        bucket: str,
        new_col: str | None = None,
    ) -> pd.DataFrame:
        out, generated_col = self._bucket_date(
            df,
            date_col,
            bucket,
        )

        if new_col and new_col != generated_col:
            out = out.rename(
                columns={generated_col: new_col}
            )

        return out

    def select_top_entities(
        self,
        df: pd.DataFrame,
        entity_col: str,
        metric_col: str,
        n: int,
        aggregate: str = "sum",
        ascending: bool = False,
        filters: list[dict] | None = None,
        keep_all_rows: bool = True,
    ) -> pd.DataFrame:
        
        self._validate_columns(df,[entity_col, metric_col])
        if aggregate not in self.ALLOWED_AGGS:
            raise ValueError(f"Unsupported aggregate: {aggregate}")

        if n <= 0:
            raise ValueError("n must be greater than zero")
        
        ranking_data = df.copy()

        if filters:
            ranking_data = self.filter_rows(
                ranking_data,
                filters,
            )

        ranking = (
            ranking_data
            .groupby(entity_col, dropna=False, as_index=False)[metric_col]
            .agg(aggregate)
            .sort_values(metric_col, ascending=ascending)
            .head(n)
        )

        selected_entities = ranking[entity_col].tolist()

        if keep_all_rows:
            return df[df[entity_col].isin(selected_entities)].copy()

        return ranking



    def run_plot_tool(
        self,
        tool_name: str,
        df: pd.DataFrame,
        args: dict[str, Any],
    ) -> dict:
        plotting_tools = {
            "plot_bar_chart": self.plot_bar_chart,
            "plot_line_chart": self.plot_line_chart,
            "plot_pie_chart": self.plot_pie_chart,
            "plot_scatter_chart": self.plot_scatter_chart,
            "plot_list": self.plot_list,
        }

        if tool_name not in plotting_tools:
            raise ValueError(f"Unknown plot tool: {tool_name}")

        args = self._filter_args(tool_name, args)
        return plotting_tools[tool_name](df=df, **args)

    def format_label(self, name: str) -> str:
        """
        Convert column-like names to display labels.

        Examples:
        - year_quarter -> Year quarter
        - percentage_contribution -> Percentage contribution
        - TOTAL_WATER_INJECTION_VOLUME -> Total water injection volume
        """
        if name is None:
            return ""

        text = str(name).replace("_", " ").strip().lower()
        return text[:1].upper() + text[1:]

    def _as_list(self, value):
        if value is None:
            return []
        return [value] if isinstance(value, str) else list(value)

    def _strip_markdown_json(self, text: str) -> str:
        """
        Remove markdown code fences from LLM JSON responses.

        Examples:
        ```json
        {...}
        ```

        ->
        {...}
        """

        text = text.strip()

        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

        return text.strip()

    def _validate_columns(
        self,
        df: pd.DataFrame,
        columns: list[str],
        label: str = "column",
    ):
        """
        Validate that all requested columns exist in the dataframe.
        """

        missing = [c for c in columns if c not in df.columns]

        if missing:
            raise ValueError(f"Missing {label}(s): {missing}")

    def _filter_args(
        self,
        tool_name: str,
        args: dict[str, Any],
    ) -> dict[str, Any]:
        """
        Remove unsupported arguments generated by the LLM.
        """

        allowed_args = {
            "plot_bar_chart": {
                "x",
                "y",
                "aggregate",
                "group_by",
                "color_by",
                "orientation",
                "barmode",
                "title",
                "template",
            },
            "plot_line_chart": {
                "x",
                "y",
                "aggregate",
                "group_by","series_by",
                "color_by",
                "date_bucket",
                "cumulative",
                "title",
                "template",
            },
            "plot_pie_chart": {
                "labels",
                "values",
                "aggregate",
                "group_by",
                "title",
                "hole",
                "template",
            },
            "plot_scatter_chart": {
                "x",
                "y",
                "color_by",
                "size_by",
                "text_by",
                "title",
                "template",
            },
            "plot_list": {
                "columns",
                "sort_by",
                "sort_order",
                "limit",
                "title",
            },
        }

        if tool_name not in allowed_args:
            raise ValueError(f"Unknown tool: {tool_name}")

        return {
            k: v
            for k, v in args.items()
            if k in allowed_args[tool_name]
        }

    def _aggregate(
        self,
        df: pd.DataFrame,
        group_by: list[str],
        value_cols: list[str],
        aggregate: str,
    ) -> pd.DataFrame:
        if aggregate not in self.ALLOWED_AGGS:
            raise ValueError(f"Unsupported aggregate: {aggregate}")

        self._validate_columns(df, group_by, "group_by column")
        self._validate_columns(df, value_cols, "value column")

        return (
            df.groupby(group_by, dropna=False, as_index=False)[value_cols]
            .agg(aggregate)
        )

    def _bucket_date(
        self,
        df: pd.DataFrame,
        date_col: str,
        bucket: str,
    ) -> tuple[pd.DataFrame, str]:
        """
        Create a date bucket column.

        bucket:
        - "D": day
        - "W": week
        - "M": month
        - "Q": quarter
        - "Y": year
        """

        out = df.copy()
        bucket_col = f"{date_col}_{bucket}"

        self._validate_columns(out, [date_col])

        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

        if bucket == "D":
            out[bucket_col] = out[date_col].dt.to_period("D").dt.to_timestamp()
        elif bucket == "W":
            out[bucket_col] = out[date_col].dt.to_period("W").dt.start_time
        elif bucket == "M":
            out[bucket_col] = out[date_col].dt.to_period("M").dt.to_timestamp()
        elif bucket == "Q":
            out[bucket_col] = out[date_col].dt.to_period("Q").dt.to_timestamp()
        elif bucket == "Y":
            out[bucket_col] = out[date_col].dt.to_period("Y").dt.to_timestamp()
        else:
            raise ValueError("date_bucket must be one of: D, W, M, Q, Y")

        return out, bucket_col

    def plot_bar_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        color_by: str | None = None,
        orientation: str = "v",
        barmode: str = "group",
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        required = [x, *y_cols]
        if color_by:
            required.append(color_by)

        self._validate_columns(df, required)

        work = df.copy()

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [x]

            if x not in group_cols:
                group_cols.insert(0, x)

            if color_by and color_by not in group_cols:
                group_cols.append(color_by)

            work = self._aggregate(work, group_cols, y_cols, aggregate)

        data = []
        groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

        for group_value, g in groups:
            for y_col in y_cols:
                if group_value is None:
                    name = self.format_label(y_col)
                else:
                    name = self.format_label(str(group_value))

                if group_value is not None and len(y_cols) > 1:
                    name = f"{self.format_label(str(group_value))} - {self.format_label(y_col)}"

                trace = {
                    "type": "bar",
                    "name": name,
                    "orientation": orientation,
                }

                if orientation == "h":
                    trace["x"] = g[y_col].tolist()
                    trace["y"] = g[x].astype(str).tolist()
                else:
                    trace["x"] = g[x].astype(str).tolist()
                    trace["y"] = g[y_col].tolist()

                data.append(trace)

        y_label = self.format_label(", ".join(y_cols))
        x_label = self.format_label(x)

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": title or f"{y_label} by {x_label}"
                },
                "xaxis": {
                    "title": {
                        "text": y_label if orientation == "h" else x_label
                    }
                },
                "yaxis": {
                    "title": {
                        "text": x_label if orientation == "h" else y_label
                    }
                },
                "barmode": barmode,
                # "template": template,
            },
            "config": {
                "responsive": True,
                "displaylogo": False,
            },
        }

    def plot_line_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        series_by: str | None = None,
        color_by: str | None = None,  # backwards compatibility
        date_bucket: str | None = None,
        cumulative: bool = False,
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        if series_by is None:
            series_by = color_by

        required = [x, *y_cols]
        if series_by:
            required.append(series_by)

        self._validate_columns(df, required)

        work = df.copy()
        x_plot = x

        if date_bucket is not None:
            work, x_plot = self._bucket_date(work, x, date_bucket)

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [x_plot]

            if x_plot not in group_cols:
                group_cols.insert(0, x_plot)

            if series_by and series_by not in group_cols:
                group_cols.append(series_by)

            work = self._aggregate(work, group_cols, y_cols, aggregate)

        sort_cols = [series_by, x_plot] if series_by else [x_plot]
        work = work.sort_values(sort_cols)

        if cumulative:
            if series_by:
                for col in y_cols:
                    work[col] = work.groupby(series_by, dropna=False)[col].cumsum()
            else:
                for col in y_cols:
                    work[col] = work[col].cumsum()

        data = []
        groups = work.groupby(series_by, dropna=False) if series_by else [(None, work)]

        total_points = len(work) * len(y_cols)
        disable_all_markers = total_points > 2000

        for group_value, g in groups:
            for y_col in y_cols:
                if group_value is None:
                    name = self.format_label(y_col)
                elif len(y_cols) == 1:
                    name = str(group_value)
                else:
                    name = f"{group_value} - {self.format_label(y_col)}"

                data.append({
                    "type": "scatter",
                    "mode": "lines",
                    "x": g[x_plot].tolist(),
                    "y": g[y_col].tolist(),
                    "name": name,
                })

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": self.format_label(title) or f"{', '.join(y_cols)} over {x}"
                },
                "xaxis": {
                    "title": {
                        "text": self.format_label(x)
                    }
                },
                "yaxis": {
                    "title": {
                        "text": self.format_label(", ".join(y_cols))
                    }
                },
            },
            "config": self.plotly_config,
        }

    def plot_pie_chart(
        self,
        df: pd.DataFrame,
        labels: str,
        values: str,
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        title: str | None = None,
        hole: float = 0.0,
        # template: str = "plotly_white",
    ) -> dict:
        self._validate_columns(df, [labels, values])

        work = df.copy()

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [labels]

            if labels not in group_cols:
                group_cols.insert(0, labels)

            work = self._aggregate(work, group_cols, [values], aggregate)

        return {
            "data": [
                {
                    "type": "pie",
                    "labels": work[labels].astype(str).tolist(),
                    "values": work[values].tolist(),
                    "hole": hole,
                }
            ],
            "layout": {
                "title": {
                    "text": title or f"{values} share by {labels}"
                },
                # "template": template,
            },
            "config": self.plotly_config
        }

    def plot_scatter_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        color_by: str | None = None,
        size_by: str | None = None,
        text_by: str | None = None,
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        required = [x, *y_cols]
        if color_by:
            required.append(color_by)
        if size_by:
            required.append(size_by)
        if text_by:
            required.append(text_by)

        self._validate_columns(df, required)

        work = df.copy()
        data = []
        groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

        total_points = len(work) * len(y_cols)
        disable_all_markers = total_points > 2000

        for group_value, g in groups:
            for y_col in y_cols:
                name = y_col if group_value is None else str(group_value)

                if group_value is not None and len(y_cols) > 1:
                    name = f"{group_value} - {y_col}"

                n_points = len(g)

                use_markers = (
                    not disable_all_markers
                    and n_points <= 100
                )

                mode = "markers" if use_markers else "lines"

                trace = {
                    "type": "scattergl",
                    "mode": mode,
                    "x": g[x].tolist(),
                    "y": g[y_col].tolist(),
                    "name": name,
                }

                if size_by:
                    size_values = pd.to_numeric(g[size_by], errors="coerce").fillna(0)
                    max_size = max(float(size_values.max()), 1.0)

                    trace["marker"] = {
                        "size": size_values.tolist(),
                        "sizemode": "area",
                        "sizeref": max_size / 40,
                        "sizemin": 4,
                    }

                if text_by:
                    trace["text"] = g[text_by].astype(str).tolist()
                    trace["hovertemplate"] = (
                        f"{x}: %{{x}}<br>"
                        f"{y_col}: %{{y}}<br>"
                        f"{text_by}: %{{text}}"
                        "<extra></extra>"
                    )

                data.append(trace)

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": title or f"{', '.join(y_cols)} vs {x}"
                },
                "xaxis": {
                    "title": {
                        "text": x
                    }
                },
                "yaxis": {
                    "title": {
                        "text": ", ".join(y_cols)
                    }
                },
                # "template": template,
            },
            "config": self.plotly_config
        }

    def plot_list(
        self,
        df: pd.DataFrame,
        columns: list[str] | str | None = None,
        *,
        sort_by: str | None = None,
        sort_order: str = "desc",
        limit: int | None = None,
        title: str | None = None,
    ) -> dict:
        work = df.copy()

        if columns is not None:
            columns = self._as_list(columns)
            self._validate_columns(work, columns)
            work = work[columns]

        if sort_by is not None:
            self._validate_columns(work, [sort_by])

            ascending = sort_order.lower() == "asc"
            work = work.sort_values(sort_by, ascending=ascending)

        if limit is not None:
            work = work.head(limit)

        header_values = [self.format_label(c) for c in work.columns]

        cell_values = []
        for col in work.columns:
            s = work[col]

            if pd.api.types.is_datetime64_any_dtype(s):
                values = s.dt.strftime("%Y-%m-%d").fillna("").tolist()
            else:
                values = s.fillna("").astype(str).tolist()

            cell_values.append(values)

        return {
            "data": [
                {
                    "type": "table",
                    "header": {
                        "values": header_values,
                        "align": "left",
                    },
                    "cells": {
                        "values": cell_values,
                        "align": "left",
                    },
                }
            ],
            "layout": {
                "title": {
                    "text": self.format_label(title) or "Table"
                },
            },
            "config": self.plotly_config
        }
    

class PresenterConfig:

    prompt :str = chart_agent_prompt
    split_subinstructions_prompr: str = split_subinstructions_prompt 
    small_table_prompt: str = small_table_prompt 
    
    def __init__(
        self,
        charting_tools: PresenterChartingTools | None = None,
    ):
        self.charting_tools = charting_tools or PresenterChartingTools()


class PresenterComponent4:
    """
    Converts an ExecutorState into UI display items.

    Each TaskResult is processed as a whole:
    - all TextResult and DataFrameResult objects are added to one context;
    - one LLM call splits the task instruction into sub-instructions;
    - each sub-instruction is associated with one source;
    - text sources become text UIItems;
    - table sources are passed to the existing dataframe presentation logic.
    """

    def __init__(
        self,
        llm: Any,
        config: PresenterConfig | None = None,
    ):
        self.llm = llm
        self.config = config or PresenterConfig()
        self.charting_tools = self.config.charting_tools

    def run(self, result_state: ExecutorState) -> PresenterResponse:
        return self.process_task_results(result_state)

    def _make_clarification_item(
        self,
        clarification_request: str,
    ) -> UIItem:
        return UIItem(
            id=f"question_{uuid4().hex[:8]}",
            type="question",
            title="Additional information required",
            data={"question": clarification_request},
        )

    def _make_text_item(
        self,
        data_result: TextResult,
    ) -> UIItem:
        return UIItem(
            id=f"text_{uuid4().hex[:8]}",
            type="text",
            title=None,
            data={"text": data_result.text},
        )

    def _make_error_item(
        self,
        task_result: TaskResult,
        data_result: object | None = None,
    ) -> UIItem:
        return UIItem(
            id=f"error_{uuid4().hex[:8]}",
            type="error",
            title="Presentation error",
            data={
                "message": f"No presenter for result from {task_result.agent}",
                "details": (
                    str(type(data_result))
                    if data_result is not None
                    else task_result.instruction
                ),
            },
        )

    def build_task_result_context(
        self,
        task_result: TaskResult,
    ) -> tuple[str, dict[str, TextResult | DataFrameResult]]:
        """
        Build:
        - one text context containing all TextResult and DataFrameResult objects;
        - a source map used later to recover the original result objects.
        """
        processor = TableResponseProcessor()

        context_parts: list[str] = []
        source_map: dict[str, TextResult | DataFrameResult] = {}

        for n, data_result in enumerate(task_result.data_results):

            if isinstance(data_result, TextResult):
                source_id = f"text_{n}"

                context_parts.append(
                    "\n".join([
                        f"SOURCE_ID: {source_id}",
                        "SOURCE_TYPE: text",
                        "CONTENT:",
                        data_result.text,
                    ])
                )

                source_map[source_id] = data_result

            elif isinstance(data_result, DataFrameResult):
                source_id = data_result.table_name
                table_context = processor.extract_table_context(data_result)

                context_parts.append(
                    "\n".join([
                        f"SOURCE_ID: {source_id}",
                        "SOURCE_TYPE: table",
                        table_context,
                    ])
                )

                source_map[source_id] = data_result

        context_text = "\n\n---\n\n".join(context_parts)

        return context_text, source_map

    def get_subinstructions(
        self,
        task_result: TaskResult,
        context_text: str,
    ) -> SubInstructions:
        """
        Split the task instruction and associate each sub-instruction
        with one available source.
        """
        messages = [
            SystemMessage(content=self.config.split_subinstructions_prompr),
            HumanMessage(
                content=(
                    f"INSTRUCTION\n"
                    f"{task_result.instruction}\n\n"
                    f"AVAILABLE SOURCES\n"
                    f"{context_text}"
                )
            ),
        ]

        structured_llm = self.llm.with_structured_output(SubInstructions)

        return structured_llm.invoke(messages)

    def process_single_task_result(
        self,
        task_result: TaskResult,
    ) -> list[UIItem]:
        context_text, source_map = self.build_task_result_context(
            task_result
        )

        sub_instructions = self.get_subinstructions(
            task_result=task_result,
            context_text=context_text,
        )

        ui_items: list[UIItem] = []

        for item in sub_instructions.items:
            source = source_map[item.source_id]

            if item.kind == "text":
                ui_items.append(
                    self._make_text_item(source)
                )

            elif item.kind == "table":
                ui_items.append(
                    self._process_dataframe(
                        source,
                        item.sub_instruction,
                    )
                )

        return ui_items

    def process_task_results(
        self,
        execution_state: ExecutorState,
    ) -> PresenterResponse:
        ui_items: list[UIItem] = []

        clarification_request = execution_state.get(
            "clarification_request"
        )

        if clarification_request:
            ui_items.append(
                self._make_clarification_item(
                    clarification_request
                )
            )

            return PresenterResponse(items=ui_items)

        for task_result in execution_state.get("task_results", []):
            ui_task_items = self.process_single_task_result(
                task_result
            )

            ui_items.extend(ui_task_items)

        return PresenterResponse(items=ui_items)

    def _present_very_small_table(
        self,
        df: pd.DataFrame,
        data_result: DataFrameResult,
        instruction: str,
    ) -> UIItem:
        data_string = df.to_json()

        print('processing very small table')
        prompt = (
            self.config.small_table_prompt
            + "\n\n"
            + (
                "### Context\n"
                f"- Table Name: {getattr(data_result, 'table_name', 'N/A')}\n"
                f"- Description: "
                f"{getattr(data_result, 'description', 'No description provided.')}\n\n"
                "### Data\n"
                f"{data_string}\n\n"
                "User question:\n"
                f"{instruction}\n"
            )
        )

        response = self.llm.invoke(prompt)

        text_output = (
            response.content
            if hasattr(response, "content")
            else str(response)
        )

        return UIItem(
            id=f"text_{uuid4().hex[:8]}",
            type="text",
            title=None,
            data={"text": text_output},
        )

    def _make_chart_item(
        self,
        figure_title: str,
        plotly_json_figure: dict,
        description: str | None = None,
    ) -> UIItem:
        return UIItem(
            id=f"chart_{uuid4().hex[:8]}",
            type="chart",
            title=figure_title,
            data={
                "engine": "plotly",
                "plotly": plotly_json_figure,
            },
            meta={
                "description": description,
            },
        )


    def _run_chart_plan(
        self,
        plan: dict,
        df: pd.DataFrame,
    ) -> dict | None:
        work = df.copy()

        preprocess_steps = plan.get("preprocess") or []
        plot = plan.get("plot")

        if preprocess_steps:
            work = self.charting_tools.run_preprocess(
                work,
                preprocess_steps,
            )

        if not plot:
            return None

        tool_name = plot.get("tool")
        args = plot.get("args") or {}

        return self.charting_tools.run_plot_tool(
            tool_name=tool_name,
            df=work,
            args=args,
        )


    def _format_label(
        self,
        name: str,
    ) -> str:
        if name is None:
            return ""

        text = str(name).replace("_", " ").strip().lower()

        return text[:1].upper() + text[1:]

    def _process_dataframe(
        self,
        data_result: DataFrameResult,
        instruction: str,
    ) -> UIItem:
        df = data_result.dataframe
        nrows, ncols = df.shape

        if nrows <= 2 and ncols <= 2:
            return self._present_very_small_table(
                df,
                data_result,
                instruction,
            )

        processor = TableResponseProcessor()

        table_context = processor.extract_table_context(
            data_result
        )

        chart_plan = self._select_chart_plan(
            instruction,
            table_context,
        )

        print('****chart plan****')
        print(instruction)
        print(chart_plan)


        chart_output = self._run_chart_plan(
            chart_plan,
            df,
        )

        return self._make_chart_item(
            self._format_label(data_result.table_name),
            chart_output,
            data_result.description,
        )

    def _strip_markdown_json(
        self,
        text: str,
    ) -> str:
        text = text.strip()

        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
        )

        text = re.sub(
            r"\s*```$",
            "",
            text,
        )

        return text.strip()

    def _select_chart_plan(
        self,
        user_query: str,
        table_context: str,
    ) -> dict:
        messages = [
            SystemMessage(content=self.config.prompt),
            HumanMessage(
                content=(
                    f"USER QUERY\n"
                    f"{user_query}\n\n"
                    f"TABLE\n"
                    f"{table_context}"
                )
            ),
        ]

        response = self.llm.invoke(messages)

        text = self._strip_markdown_json(
            response.content
        )

        return json.loads(text)
    

    

zero temp, seed 42, top_p = 1
****chart plan****
Identify the top 5 producer wells based on their cumulative oil production in 2018.
{'reason': 'The table already contains the top 5 producer wells based on their cumulative oil production in 2018, so no preprocessing is needed.', 'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'total_oil_volume', 'aggregate': None, 'group_by': ['NAME'], 'color_by': None, 'orientation': 'v', 'barmode': 'group', 'title': 'Top 5 Producer Wells by Cumulative Oil Production in 2018'}}}
****chart plan****
Plot the yearly liquid production for the top 5 producer wells.
{'reason': 'The table already contains yearly liquid production data for the top 5 producer wells, so no preprocessing is needed.', 'preprocess': [], 'plot': {'tool': 'plot_line_chart', 'args': {'x': 'YEAR', 'y': 'total_liquid_volume', 'aggregate': None, 'group_by': ['YEAR', 'NAME'], 'series_by': 'NAME', 'date_bucket': None, 'cumulative': False, 'title': 'Yearly L

[UIItem(id='text_6d571369', type='text', title=None, data={'text': 'Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. \n\nA VRR of 1.0 indicates full replacement, helping to sustain pressure and improve sweep efficiency, while a VRR less than 1.0 suggests under-replacement, potentially leading to pressure decline. Conversely, a VRR greater than 1.0 may indicate over-injection, which could lead to operational inefficiencies or formation damage.'}, meta={}),
 UIItem(id='chart_c408deeb', type='chart', title='Top 5 producers 2018', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'bar', 'name': 'Total oil volume', 'orientation': 'v', 'x': ['P4', 'P3', 'P1', 'P2'], 'y': [12875.199035

Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. 

A VRR of 1.0 indicates full replacement, helping to sustain pressure and improve sweep efficiency, while a VRR less than 1.0 suggests under-replacement, potentially leading to pressure decline. Conversely, a VRR greater than 1.0 may indicate over-injection, which could lead to operational inefficiencies or formation damage.


In [ ]:
#pprint.pprint( execution_state['task_results'][1] )
t = execution_state['task_results'][1]
instruction = t.instruction 
#t.data_results.insert(0, TextResult(text="All the fruits in the basket are sweet."))

print( instruction )
p = TableResponseProcessor()
context = [] 

aux= {} 
for n,data_result in enumerate(t.data_results):


    if isinstance( data_result, TextResult):
        print("processing text ")
        context.append( data_result.text )
        aux[n] = data_result.text 

    if isinstance(data_result, DataFrameResult):
        print("processing dataframe result")
        table_context = p.extract_table_context( data_result )
        context.append( table_context )
        aux[ data_result.table_name ] = (table_context,data_result)
    
prompt = """You will receive an 'instruction' and information of tables and text. your job is to 
analyze the instruction. It might contain several sub-instructions. 
Decide what parts of the information available can be used to execute the 
instruction and its sub-instructions. 

Do not explain anything, do not add more details than strictly needed to produce the required output 
"""
class SubInstruction(BaseModel):
    sub_instruction: str = Field(
        description="The specific part of the instruction."
    )
    kind:  Literal['table','text']
    
    data: str  = Field(
        description="The name of the table or the textual information "
    )
  
  
class SubInstructions(BaseModel):

    items: List[SubInstruction] =  Field(description="The list of sub-instructions and the information relevant for each")

context = "\n\n".join(context)

#instruction2 = "Check if the fruits are sour or sweet, " + instruction# then list the top 5 producers by cummulated oil produced in 2018 "

context =  "\n" + context + "\n\n" + instruction + "\n\n"
messages = [SystemMessage(prompt + "\n\n" + context )]

structured_llm = llm.with_structured_output(SubInstructions)
response = structured_llm.invoke(messages)

In [ ]:
pprint.pprint(context)


In [ ]:
pprint.pprint(response.items)


In [ ]:

items = [] 

for i in response.items:
    if i.kind=='text':
        print("It is a text")
        r =  UIItem(
                id=f"text_{uuid4().hex[:8]}",
                type="text",
                title=None,
                data={"text": i.data}
            )
        items.append( r )

    if i.kind=='table':

        #print( '**',i.sub_instruction,'**')
        table_name = i.data 
        acontext = aux[table_name ][0]
        data_result = aux[table_name][1]
        #print('table', table_name, acontext )
        
        r = presenter._process_dataframe(data_result,i.sub_instruction)
        items.append( r )

        


In [ ]:
print( items )

print()
print()
print()

for item in items:

    if item.type=='text':
        print( item.data['text'])

    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


## More organized

In [ ]:
from typing import Literal
from uuid import uuid4

from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage


# =============================================================================
# Structured output models
# =============================================================================

class SubInstruction(BaseModel):
    sub_instruction: str = Field(
        description=(
            "The specific part of the original instruction that must be "
            "presented using the selected source."
        )
    )

    kind: Literal["table", "text"] = Field(
        description="The type of source associated with this sub-instruction."
    )

    source_id: str = Field(
        description=(
            "The exact SOURCE ID provided in the available sources. "
            "For a table, this is the exact table name. "
            "For text, this is the exact text result identifier."
        )
    )


class SubInstructions(BaseModel):
    items: list[SubInstruction] = Field(
        description=(
            "The requested outputs, in the order in which they should be "
            "presented. Irrelevant and intermediate sources must be omitted."
        )
    )


# =============================================================================
# Routing prompt
# =============================================================================

TASK_PRESENTATION_ROUTER_PROMPT = """
You receive one instruction and a set of available sources.

The instruction may contain several sub-instructions.

Your job is to:

1. Identify each distinct output explicitly requested by the instruction.
2. Match each requested output to exactly one relevant source.
3. Return the outputs in the order in which they should be presented.
4. Use the exact SOURCE ID provided for each source.
5. Omit sources that are irrelevant or only intermediate calculation results.
6. Do not invent facts, tables, source IDs, calculations, or additional requests.
7. Do not explain your decisions.
8. Do not create an item when the available sources cannot support it.
9. Do not repeat the same source unless it is genuinely required for two
   different requested outputs.

For a text source:
- Use it when the source directly contains the requested textual answer.
- The sub_instruction should describe the part of the instruction answered
  by the text.
- The source_id must be the exact text SOURCE ID.

For a table source:
- Use it when the table contains the information required for the requested
  table, chart, list, ranking, comparison, or numerical presentation.
- The sub_instruction must contain only the part of the original instruction
  that the selected table can address.
- The source_id must be the exact table SOURCE ID.

Important:
- A table used only to calculate another final table is usually an intermediate
  source and should be omitted unless the user explicitly requested it.
- Do not return the source content itself.
- Return only the structured result.
"""


# =============================================================================
# Select the TaskResult to process
# =============================================================================

task_result_index = 1

task_result = execution_state["task_results"][task_result_index]
instruction = task_result.instruction

print("TASK INSTRUCTION")
print(instruction)
print()


# =============================================================================
# Build source context and source lookup
# =============================================================================

table_processor = TableResponseProcessor()

source_contexts: list[str] = []
source_lookup: dict[str, TextResult | DataFrameResult] = {}

for result_index, data_result in enumerate(task_result.data_results):

    if isinstance(data_result, TextResult):
        source_id = f"text_result_{result_index}"

        source_contexts.append(
            "\n".join([
                f"SOURCE ID: {source_id}",
                "SOURCE KIND: text",
                "CONTENT:",
                data_result.text,
            ])
        )

        source_lookup[source_id] = data_result

    elif isinstance(data_result, DataFrameResult):
        source_id = data_result.table_name
        table_context = table_processor.extract_table_context(data_result)

        source_contexts.append(
            "\n".join([
                f"SOURCE ID: {source_id}",
                "SOURCE KIND: table",
                table_context,
            ])
        )

        source_lookup[source_id] = data_result

    else:
        print(
            "Ignoring unsupported data result:",
            type(data_result).__name__,
        )


available_sources_context = "\n\n---\n\n".join(source_contexts)

print("AVAILABLE SOURCE IDS")
for source_id, source in source_lookup.items():
    print(f"- {source_id}: {type(source).__name__}")
print()


# =============================================================================
# One LLM call to split the instruction and route each part to a source
# =============================================================================

messages = [
    SystemMessage(content=TASK_PRESENTATION_ROUTER_PROMPT),
    HumanMessage(
        content=(
            f"ORIGINAL INSTRUCTION\n"
            f"{instruction}\n\n"
            f"AVAILABLE SOURCES\n"
            f"{available_sources_context}"
        )
    ),
]

structured_llm = llm.with_structured_output(SubInstructions)
routing_response = structured_llm.invoke(messages)

print("ROUTING RESPONSE")
for routed_item in routing_response.items:
    print(routed_item)
print()


# =============================================================================
# Convert the routed outputs into UIItems
# =============================================================================

items: list[UIItem] = []

for routed_item in routing_response.items:
    source = source_lookup.get(routed_item.source_id)

    if source is None:
        items.append(
            UIItem(
                id=f"error_{uuid4().hex[:8]}",
                type="error",
                title="Presentation error",
                data={
                    "message": (
                        "The presentation router selected an unknown source."
                    ),
                    "details": routed_item.source_id,
                },
            )
        )
        continue

    if routed_item.kind == "text":
        if not isinstance(source, TextResult):
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            "The presentation router classified a non-text "
                            "source as text."
                        ),
                        "details": routed_item.source_id,
                    },
                )
            )
            continue

        items.append(
            UIItem(
                id=f"text_{uuid4().hex[:8]}",
                type="text",
                title=None,
                data={
                    "text": source.text,
                },
                meta={
                    "sub_instruction": routed_item.sub_instruction,
                    "source_id": routed_item.source_id,
                },
            )
        )

    elif routed_item.kind == "table":
        if not isinstance(source, DataFrameResult):
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            "The presentation router classified a non-table "
                            "source as a table."
                        ),
                        "details": routed_item.source_id,
                    },
                )
            )
            continue

        try:
            ui_item = presenter._process_dataframe(
                source,
                routed_item.sub_instruction,
            )

            if ui_item is not None:
                ui_item.meta = {
                    **ui_item.meta,
                    "sub_instruction": routed_item.sub_instruction,
                    "source_id": routed_item.source_id,
                }
                items.append(ui_item)

        except Exception as exc:
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            f"Could not present table "
                            f"{routed_item.source_id}."
                        ),
                        "details": str(exc),
                    },
                    meta={
                        "sub_instruction": routed_item.sub_instruction,
                        "source_id": routed_item.source_id,
                    },
                )
            )


# =============================================================================
# Final presenter response
# =============================================================================

presenter_response = PresenterResponse(items=items)

print("GENERATED UI ITEMS")
for item in presenter_response.items:
    print(
        {
            "id": item.id,
            "type": item.type,
            "title": item.title,
            "source_id": item.meta.get("source_id"),
            "sub_instruction": item.meta.get("sub_instruction"),
        }
    )

presenter_response
  

In [ ]:
items = presenter_response.items 
print( items )

print()
print()
print()

for item in items:

    if item.type=='text':
        print( item.data['text'])

    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


In [ ]:


presenter = PresenterComponent4( llm )
#presenter = PresenterComponent2( llm )

ui_items = presenter.run( execution_state )
ui_items

In [ ]:
ui_items.items

In [ ]:
item = ui_items.items[1]
item = item.data['plotly']
pio.show(item)

item = ui_items.items[2]
item = item.data['plotly']
pio.show(item)

item = ui_items.items[3]
item = item.data['plotly']
pio.show(item)